## *R6. HMM Parameters: States, Observations, Initial Distribution, and Emission Matrix*

In this section, we extend the project's Markov chain formulation into a *Hidden Markov Model (HMM)* so that unobservable student study states can be linked to observable academic outcomes across time. To stay consistent with the earlier sections of the notebook, we reuse the shared state definitions and transition dynamics (STATES and P) and introduce the emission structure (E).

### Core HMM Definitions and Alignment with Earlier Sections

The hidden states (STATES) are Focused, Distracted, and Burned Out, representing the latent study states of a student that are not directly observed. The observable outcomes (OBSERVATIONS) are High Score, Average Score, and Low / Missed, representing the recorded performance results at each step.

The initial state distribution (pi_0) is [0.50, 0.30, 0.20], matching the initial distribution used earlier in the project and indicating higher initial likelihood of starting in Focused. The transition matrix (P) is aligned with the transition probabilities from the preceding sections (with a fallback definition only if the cell is run standalone), ensuring full compatibility across team components.

### Emission Matrix (E) Design and Validation

The emission matrix E gives the conditional probability of observing each outcome given the current hidden state, where every row is non-negative and sums to 1.00:

- For Focused ([0.70, 0.20, 0.10]), probabilities lean strongly toward High Score (0.70), with smaller chances for Average Score (0.20) and Low / Missed (0.10).
- For Distracted ([0.25, 0.45, 0.30]), probabilities center around Average Score (0.45), alongside moderate chances for Low / Missed (0.30) and High Score (0.25).
- For Burned Out ([0.10, 0.25, 0.65]), probabilities skew heavily toward Low / Missed (0.65), with lower chances for Average Score (0.25) and High Score (0.10).

We also include assert checks to verify that pi_0 sums to 1.0 and that every row of both P and E sums to 1.0.

> *Note:* The probabilities in E are illustrative domain assumptions created for modeling and demonstration in this project rather than empirical measurements from real students.

In [9]:
import numpy as np
import pandas as pd

# R6: Define hidden states, observations, initial distribution, and emission matrix
STATES = ["Focused", "Distracted", "Burned Out"]
OBSERVATIONS = ["High Score", "Average Score", "Low / Missed"]

obs_to_idx = {obs: idx for idx, obs in enumerate(OBSERVATIONS)}

pi_0 = np.array([0.50, 0.30, 0.20])

# Ensure P is available from earlier cells; fallback definition if running cell standalone
if "P" not in globals():
    P = np.array([
        [0.70, 0.20, 0.10],
        [0.30, 0.50, 0.20],
        [0.20, 0.35, 0.45]
    ])

E = np.array([
    [0.70, 0.20, 0.10],
    [0.25, 0.45, 0.30],
    [0.10, 0.25, 0.65]
])

assert np.isclose(pi_0.sum(), 1.0), "pi_0 must sum to 1."
assert np.allclose(P.sum(axis=1), 1.0), "Each row of P must sum to 1."
assert np.allclose(E.sum(axis=1), 1.0), "Each row of E must sum to 1."
E_df = pd.DataFrame(E, index=STATES, columns=OBSERVATIONS)
print("Emission Matrix E:")
display(E_df)

Emission Matrix E:


,High Score,Average Score,Low / Missed
Focused,0.70,0.20,0.10
Distracted,0.25,0.45,0.30
Burned Out,0.10,0.25,0.65


## R7. Forward Algorithm and Observation Sequence Probability

With the HMM parameters (STATES, OBSERVATIONS, pi_0, P, and E) defined, we implement the *Forward Algorithm* (forward_algorithm) to compute the exact probability of observing a given sequence of outcomes (obs_seq) over time. Instead of enumerating every possible hidden state path individually, the algorithm accumulates forward variables (alpha) step by step from t = 0 to T_len - 1.

### Forward Algorithm Steps

In *Initialization (t = 0)*, we compute the probability of starting in hidden state j and observing the first outcome obs_seq[0]:

alpha[0, j] = pi_0[j] * E[j, obs_0]

In *Induction (t from 1 to T_len - 1)*, for each step t and state j, we sum the incoming weighted probabilities from all previous states i using transition matrix P, then multiply by the emission probability of observing obs_t from state j:

alpha[t, j] = sum_i(alpha[t - 1, i] * P[i, j]) * E[j, obs_t]

In *Termination*, we sum the final forward variables alpha[-1] across all hidden states to obtain the total probability of the entire observation sequence (prob_obs):

prob_obs = sum_j(alpha[-1, j])

### Sequence Evaluation and Interpretation

Applying forward_algorithm to the sample observation sequence ["High Score", "Average Score", "Low / Missed", "Low / Missed"] gives a total sequence probability of *P(Observation Sequence) = 0.011057* (about 1.11%).

A probability around 0.011057 is expected for one specific four-step sequence because total probability mass is spread across all possible sequences of that length. Along the sequence, the initial High Score supports Focused, the Average Score shifts attention toward Distracted, and the two consecutive Low / Missed observations at the end push substantial probability weight toward Burned Out

In [7]:
# R7: Forward Algorithm
def forward_algorithm(obs_seq, pi_init, trans_mat, emit_mat):
    obs_indices = [obs_to_idx[o] for o in obs_seq]
    T_len = len(obs_indices)
    N_states = len(pi_init)

    alpha = np.zeros((T_len, N_states))

    # Initialization (t = 0)
    alpha[0] = pi_init * emit_mat[:, obs_indices[0]]

    # Induction (t = 1 to T_len - 1)
    for t in range(1, T_len):
        for j in range(N_states):
            alpha[t, j] = np.sum(alpha[t - 1] * trans_mat[:, j]) * emit_mat[j, obs_indices[t]]

    # Termination: total probability of the observation sequence
    prob_obs = np.sum(alpha[-1])
    return prob_obs, alpha

sample_obs = ["High Score", "Average Score", "Low / Missed", "Low / Missed"]
seq_prob, alpha_table = forward_algorithm(sample_obs, pi_0, P, E)

alpha_df = pd.DataFrame(
    alpha_table,
    index=[f"t={i} ({obs})" for i, obs in enumerate(sample_obs)],
    columns=STATES
)

print("Forward variables (alpha):")
display(alpha_df)
print(f"P(Observation Sequence) = {seq_prob:.6f}")

Forward variables (alpha):


,Focused,Distracted,Burned Out
t=0 (High Score),0.350000,0.075000,0.020000
t=1 (Average Score),0.054300,0.051525,0.014750
t=2 (Low / Missed),0.005642,0.012535,0.014542
t=3 (Low / Missed),0.001062,0.003746,0.006250


P(Observation Sequence) = 0.011057
